# Import

In [1]:
# Standard library imports
import os
import sys
from datetime import date

# Local application imports
sys.path.append('..')
sys.path.append('../..')

from file_management import get_files_dir, check_save_file
from clean_products import *

# Get file directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# colours
#1f9e89
#35b779
#80c066

# Input and Output files

## Input

Articles that fit into the metabolic engineering classification

In [2]:
file = 'filtered_metabolic_eng_articles_with_raw_products_V_2025_09_30.pickle'
raw_products = pd.read_pickle(
    file)

In [3]:
raw_products.shape

(16450, 12)

In [4]:
raw_products.dtypes


Title                 string[python]
Abstract              string[python]
Journal                     category
Year                           int16
PMC_ID                         Int32
DOI                   string[python]
Type                  string[python]
Author                string[python]
Text                  string[python]
Product_Source              category
Separated_products            object
Doc_text                      object
dtype: object

In [5]:
raw_products.head(60).loc[raw_products.head(60).Product_Source=='title',['Title','Product_Source','Separated_products','Doc_text']]

,Title,Product_Source,Separated_products,Doc_text
10618204,Increased production of zeaxanthin and other p...,title,zeaxanthin,production of zeaxanthin and
10653745,A novel genetically engineered pathway for syn...,title,poly(hydroxyalkanoic-acids),A genetically route for production of poly(hyd...
10689078,Metabolic engineering of Alcaligenes eutrophus...,title,polyhydroxyalkanoate,polyhydroxyalkanoate biosynthesis
10742205,Properties of engineered poly-3-hydroxyalkanoa...,title,poly-3-hydroxyalkanoates,poly-3-hydroxyalkanoates produced
10802621,Improving lycopene production in Escherichia c...,title,lycopene,lycopene production
10814415,A short total synthesis of (+)-furanomycin.,title,(+)-furanomycin,A short total production of (+)-furanomycin
10818234,Genetic evidence of branching in the isoprenoi...,title,isopentenyl diphosphate dimethylallyl diphosphate,for the production of isopentenyl diphosphate ...
10820335,Production of enantiopure styrene oxide by rec...,title,styrene oxide,production of styrene oxide
10849853,Polyhydroxybutyrate production from carbon dio...,title,polyhydroxybutyrate,polyhydroxybutyrate production
10862673,Recombinant protein production driven by the t...,title,protein,protein production


In [6]:
raw_products.head(100).loc[raw_products.head(100).Product_Source=='title',['Title','Product_Source','Separated_products','Doc_text']]

,Title,Product_Source,Separated_products,Doc_text
10618204,Increased production of zeaxanthin and other p...,title,zeaxanthin,production of zeaxanthin and
10653745,A novel genetically engineered pathway for syn...,title,poly(hydroxyalkanoic-acids),A genetically route for production of poly(hyd...
10689078,Metabolic engineering of Alcaligenes eutrophus...,title,polyhydroxyalkanoate,polyhydroxyalkanoate biosynthesis
10742205,Properties of engineered poly-3-hydroxyalkanoa...,title,poly-3-hydroxyalkanoates,poly-3-hydroxyalkanoates produced
10802621,Improving lycopene production in Escherichia c...,title,lycopene,lycopene production
10814415,A short total synthesis of (+)-furanomycin.,title,(+)-furanomycin,A short total production of (+)-furanomycin
10818234,Genetic evidence of branching in the isoprenoi...,title,isopentenyl diphosphate dimethylallyl diphosphate,for the production of isopentenyl diphosphate ...
10820335,Production of enantiopure styrene oxide by rec...,title,styrene oxide,production of styrene oxide
10849853,Polyhydroxybutyrate production from carbon dio...,title,polyhydroxybutyrate,polyhydroxybutyrate production
10862673,Recombinant protein production driven by the t...,title,protein,protein production


## Output

In [7]:
general_name = 'filtered_metabolic_eng_articles_with_products_cleaned'

today = date.today()
today = today.strftime("%Y_%m_%d")

output_file = f'{general_name}_V_{today}.json'
output_file

'filtered_metabolic_eng_articles_with_products_cleaned_V_2025_09_30.json'

# Cleanup

In [8]:
# We separated to be able to save the file as a pickle, so we join it again
raw_products
exploded =raw_products

# Rejoin by index
grouped_products = exploded.groupby(level=0)['Separated_products'].agg(list)
base_data = exploded.drop(columns=['Separated_products']).drop_duplicates()
joined_data_rejoined = base_data.join(grouped_products)
joined_data_rejoined.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Separated_products
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,title,production of zeaxanthin and,[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,abstract,heme proteins production,[heme proteins]
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,not_found,NaN,[nan]
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,abstract,ethanol production rates,[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,abstract,sterol biosynthesis,[sterol]


## Remove words or products

In [9]:
# ──  Apply the cleaning to the DataFrame ───────────────────────────────
removed_log=[]
joined_data_rejoined['Separated_products']= joined_data_rejoined.apply(
    lambda row: clean_product_list(row.name, row['Separated_products'],removed_log),
    axis=1
)



In [ ]:
import logging

# Set up logging (you can configure this at the top of your notebook)
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

if not removed_log:
    logger.info("✅ No fake products (dates, numbers, vol.#, suffix words) found.")
else:
    logger.info(f"❌ Found and removed fake products from {len(removed_log)} rows:\n")
    for entry in removed_log:
        idx = entry['row_index']
        original = entry['original_list']
        removed = entry['removed_items']
        logger.info(f"Row {idx} — original list: {original}")
        for val, reason in removed:
            logger.info(f"   ⟶ Removed: '{val}'  ({reason})")
        logger.info("")  # Empty line for separation

In [ ]:
# ── 3. Apply the cleaning to the DataFrame ───────────────────────────────
removed_log=[]
joined_data_rejoined['Separated_products'] = joined_data_rejoined.apply(
    lambda row: clean_product_list(row.name, row['Separated_products'],removed_log),
    axis=1
)

# ── 4. Report the removals with full context ─────────────────────────────

if not removed_log:
    logger.info("✅ No fake products (dates, numbers, vol.#, suffix words) found.")
else:
    logger.info(f"❌ Found and removed fake products from {len(removed_log)} rows:\n")
    for entry in removed_log:
        idx = entry['row_index']
        original = entry['original_list']
        removed = entry['removed_items']
        logger.info(f"Row {idx} — original list: {original}")
        for val, reason in removed:
            logger.info(f"   ⟶ Removed: '{val}'  ({reason})")
        logger.info("")  # Empty line for separation

In [12]:
joined_data = joined_data_rejoined

## Separate products

In [13]:
joined_data['Separated_products'] = joined_data['Separated_products'].apply(split_acid_related_products)

📎 SPLIT (Rule 3 - acid + compound): 'acid carbon' → ['acid', 'carbon']
📎 SPLIT (Rule 4 - isoamyl + ate): 'isopentenyl diphosphate dimethylallyl diphosphate' → ['isopentenyl diphosphate', 'dimethylallyl diphosphate']
📎 SPLIT (Rule 3 - acid + compound): 'fatty acid r1128' → ['fatty acid', 'r1128']
📎 SPLIT (Rule 3 - acid + compound): 'acid australian' → ['acid', 'australian']
📎 SPLIT (Rule 3 - acid + compound): 'amino acid genes' → ['amino acid', 'genes']
📎 SPLIT (Rule 3 - acid + compound): 'cmp-sialic acid glycoproteins' → ['cmp-sialic acid', 'glycoproteins']
📎 SPLIT (Rule 3 - acid + compound): 'mannitol lactic acid pyruvate' → ['mannitol lactic acid', 'pyruvate']
📎 SPLIT (Rule 4 - default ate split): 'lactate succinate' → ['lactate', 'succinate']
📎 SPLIT (Rule 4 - ate + ase): 'pyruvate carboxylase succinate' → ['pyruvate carboxylase', 'succinate']
📎 SPLIT (Rule 3 - acid + compound): '1-desmethylcobyrinic acid a,c-diamide' → ['1-desmethylcobyrinic acid', 'a,c-diamide']
📎 SPLIT (Rule 4 - 

In [14]:
joined_data['Separated_products'] = joined_data['Separated_products'].apply(split_acid_related_products)

📎 SPLIT (Rule 3 - acid + compound): 'N-acyl sialic acid analog' → ['N-acyl sialic acid', 'analog']
📎 SPLIT (Rule 3 - acid + compound): 'galactonic acid meso -' → ['galactonic acid', 'meso -']
📎 SPLIT (Rule 3 - acid + compound): 'octanoic acid tolerance' → ['octanoic acid', 'tolerance']


In [15]:
import numpy as np

joined_data['Separated_products'] = joined_data['Separated_products'].apply(
    lambda x: np.nan if isinstance(x, list) and len(x) == 1 and pd.isna(x[0]) else x
)

In [16]:
def acid_ate(lista):
    new_lista = []
    for text in lista:
        new_text = re.sub(r'ic acid\b', 'ate', text)
        new_text = re.sub(r'ate acid\b', 'ate', new_text)

        new_text = re.sub(r'\bamino\b\s?$', 'amino acid', new_text)
        #new_text = re.sub(r'\bpolycyclic tetramate\b\s?$', 'polycyclic tetramate macrolactams', new_text)

        new_lista.append(new_text)
    return new_lista

In [17]:
joined_data.Separated_products

10618204                     [zeaxanthin]
10618209                  [heme proteins]
10631776                              NaN
10649237                        [ethanol]
10649449                         [sterol]
                        ...              
40572208                    [nitrogenase]
40573728    [cucurbitane-type mogrosides]
40577193                        [ethanol]
40578703                              NaN
40579636                          [γ-pga]
Name: Separated_products, Length: 16130, dtype: object

In [18]:
# Change 'ic acid' to 'ate' and amino alone to amino acid
joined_data['Acid_normalized'] = joined_data['Separated_products'].apply(
    lambda x: acid_ate(x) if isinstance(x, list) else np.nan
)

CHECK PRECURSOR CASES 

In [19]:
joined_data['Acid_normalized'] = joined_data['Acid_normalized'].apply(split_acid_related_products)

📎 SPLIT (Rule 4 - default ate split): 'microorganisms sialate (sialate)' → ['microorganisms sialate', '(sialate)']
📎 SPLIT (Rule 4 - isoamyl + ate): 'isoamyl acetate succinate' → ['isoamyl acetate', 'succinate']
📎 SPLIT (Rule 4 - default ate split): 'methacrylate precursor 2-hydroxyisobutyrate' → ['methacrylate', 'precursor 2-hydroxyisobutyrate']
📎 SPLIT (Rule 4 - default ate split): 'dicarboxylate 2-methylsuccinate' → ['dicarboxylate', '2-methylsuccinate']
📎 SPLIT (Rule 4 - default ate split): 'lactate 3-hydroxypropionate' → ['lactate', '3-hydroxypropionate']
📎 SPLIT (Rule 4 - default ate split): 'atp-citrate itaconate' → ['atp-citrate', 'itaconate']
📎 SPLIT (Rule 4 - default ate split): 'poly(3-hydroxybutyrate)/poly(lactate)' → ['poly(3-hydroxybutyrate', ')/poly(lactate)']
📎 SPLIT (Rule 4 - default ate split): 'glutamate itaconate' → ['glutamate', 'itaconate']


In [20]:
joined_data['Acid_antibiotic_normalized'] = joined_data['Acid_normalized'].apply(split_antibiotic_related_products)

📎 EXTRACT (Rule 2 - antibiotic): 'antiparasitic megalomicin' → 'megalomicin'
📎 IGNORE (Rule 1 - context): 'peptide' in 'peptide antibiotic bacitracin'
📎 EXTRACT (Rule 2 - antibiotic): 'peptide antibiotic bacitracin' → 'bacitracin'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotic' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'peptide antibiotics' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotics' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotic' → 'antibiotic'
📎 IGNORE (Rule 1 - context): 'polyene macrolide' in 'polyene macrolide antibiotic nystatin'
📎 EXTRACT (Rule 2 - antibiotic): 'polyene macrolide antibiotic nystatin' → 'nystatin'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotics' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotic' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotic' → 'antibiotic'
📎 COLLAPSE (Rule 5 - generic antibiotic): 'antibiotic' → 'antibiotic'
📎 COLLAPSE (Rule 5 -

In [21]:
 'antibiotic (+)-asperlin'

'antibiotic (+)-asperlin'

In [22]:
top_terms = joined_data.Acid_antibiotic_normalized.explode().dropna().value_counts().head(150).index
top_terms = set(top_terms)
top_terms = top_terms- {'protein','proteins','enzyme','enzymes','aromatic','peptide','peptides',
                        'gene','growth','lipid','lipids'} #special case lipid


In [ ]:

# Now apply to your dataframe
joined_data['Acid_antibiotic_normalized_cleaned'] = joined_data['Acid_antibiotic_normalized'].apply(
    lambda x: split_if_multiple_top_terms(x, top_terms)[0] if isinstance(x, list) else x
)

In [24]:
# ── 3. Apply the cleaning to the DataFrame ───────────────────────────────
removed_log=[]
joined_data['Acid_antibiotic_normalized_cleaned'] = joined_data.apply(
    lambda row: clean_product_list(row.name, row['Acid_antibiotic_normalized_cleaned'],removed_log),
    axis=1
)

# ── 4. Report the removals with full context ─────────────────────────────

if not removed_log:
    logger.info("✅ No fake products (dates, numbers, vol.#, suffix words) found.")
else:
    logger.info(f"❌ Found and removed fake products from {len(removed_log)} rows:\n")
    for entry in removed_log:
        idx = entry['row_index']
        original = entry['original_list']
        removed = entry['removed_items']
        logger.info(f"Row {idx} — original list: {original}")
        for val, reason in removed:
            logger.info(f"   ⟶ Removed: '{val}'  ({reason})")
        logger.info("")  # Empty line for separation

✅ No fake products (dates, numbers, vol.#, suffix words) found.


In [25]:
productitos = joined_data.Acid_antibiotic_normalized_cleaned.explode().dropna().value_counts()

In [26]:
productitos_s = joined_data.Acid_antibiotic_normalized_cleaned.explode().dropna().str.strip('s').value_counts()

In [28]:
import re

# Function to find words ending with 'ate' followed by another 'ate' word
def find_double_ate(s):
    words = s.split()
    for i in range(len(words)-1):
        if words[i].endswith('ate') and words[i+1].endswith('ate'):
            return True
    return False

# Filter the subset
double_ate = productitos[productitos.index.to_series().apply(find_double_ate)]

print(double_ate)

Acid_antibiotic_normalized_cleaned
mevalonate 5-phosphate     1
polyphosphate phosphate    1
Name: count, dtype: int64


923

## Remove fake products 

In [29]:
joined_data.columns

Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text', 'Separated_products',
       'Acid_normalized', 'Acid_antibiotic_normalized',
       'Acid_antibiotic_normalized_cleaned'],
      dtype='object')

In [30]:
output_data  = joined_data.loc[:,['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI',
       'Type', 'Author', 'Text','Product_Source','Doc_text', 'Separated_products','Acid_antibiotic_normalized_cleaned']]

In [31]:
output_data.columns

Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text', 'Separated_products',
       'Acid_antibiotic_normalized_cleaned'],
      dtype='object')

In [32]:
output_file

'filtered_metabolic_eng_articles_with_products_cleaned_V_2025_09_30.json'

## Remove words from the start like human, pure etc 

In [33]:
import re
import numpy as np

# --- 1. Regex replacements ---

replacements = [
    (re.compile(r'\b(ligno)?cellulosic\b'), ''),   # remove "cellulosic" or "lignocellulosic"
    (re.compile(r'\bpure\b'), ''),                # remove "pure"
    (re.compile(r'\bhuman\b'), ''),               # remove "human"
    (re.compile(r'\b(bio)(fuel|ethanol|butanol|diesel|ethylene|lipid)s?\b'), r'\2'),  # "bioethanol" -> "ethanol"
    (re.compile(r'\bcell\s*growth\b'), 'biomass'),  # "cell growth" -> "biomass"
    (re.compile(r'\s+genes?\b'), ''),             # remove "gene" or "genes"
]

# first-word stoplist
first_word_stoplist = {
    "three", "growth", "membrane", "and/or", "polymer", "precursor",
    "thermophilic", "microalgal", "microalgae", 
}

def apply_replacements(values):
    """Apply regex replacements + remove certain first words in each element of a list."""
    if not isinstance(values, list):
        return values
    
    cleaned = []
    for v in values:
        # run regex replacements
        for pattern, repl in replacements:
            v = pattern.sub(repl, v)
        v = v.strip()

        # skip if empty
        if not v:
            continue

        # split into words, check first word
        tokens = v.split()
        if tokens and tokens[0].lower() in first_word_stoplist:
            # drop the first token
            tokens = tokens[1:]
            v = " ".join(tokens).strip()

        # add if still non-empty
        if v:
            cleaned.append(v)

    return cleaned if cleaned else np.nan

In [34]:
output_data["Acid_antibiotic_normalized_cleaned"] = output_data["Acid_antibiotic_normalized_cleaned"].apply(apply_replacements)


In [35]:
output_data.Acid_antibiotic_normalized_cleaned.dropna()

10618204                     [zeaxanthin]
10618209                  [heme proteins]
10649237                        [ethanol]
10649449                         [sterol]
10653745    [poly(hydroxyalkanoic-acids)]
                        ...              
40572067                   [amylosucrase]
40572208                    [nitrogenase]
40573728    [cucurbitane-type mogrosides]
40577193                        [ethanol]
40579636                          [γ-pga]
Name: Acid_antibiotic_normalized_cleaned, Length: 14520, dtype: object

In [36]:
output_data.columns

Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text', 'Separated_products',
       'Acid_antibiotic_normalized_cleaned'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np

# your fake values list (already deduplicated)
fake_values = [
    "towards","difficult","mixture","role","method","standpoint","certain",
    "several","future","overall","degree","suite","procedure","repertoire",
    "recovery","maincatalyst","strategy","discovery","performance",
    "superiorperformance","mechanism","response","interplay","spectrum",
    "candidate","technique","could","proxy","entire","academic","widespread",
    "healthy","costly","despite","dual","myriad","mainobstacle","bilevel",
    "heterotroph","organic","toolbox","wildtype","choice","data","toward",
    "length","survival","criterion","biology","immensevariety","latter",
    "shortcut","frontier","scaleup","bulkscale","grand","common","universal",
    "global","addedvalue","recalcitrance","storage","palette","mode",
    "productivity","machinery","attenuation","redox","phenotypic","fossil",
    "variety","vast","sensor","authentic","revision","scenario","dropin",
    "since","delivery","barrier","portfolio","disease","workhorse","result",
    "phenotype","biotic","mimicry","efficacy","unknown","covert",
    "gatekeeper","native","area","anthropogenic","formula","basic",
    "enantiomeric","problem","cheap","may","chapter","acidic","superior",
    "wealth","scope","benign","ton","juice","proven","meaningful","concise",
    "performer","rand","kind","division","resource","wastewater", "main",
] + ['lignocellulosic','cellulosic','pure','human','synthesis',
     'cell','gene','genes','food','acid']

# ensure it's a set for faster lookup
fake_set = set(fake_values)

# function to clean each row (with printing if fake found)
def clean_list_with_logging(values, doc_text, title):
    if not isinstance(values, list):  # skip NaN or non-list
        return values
    
    cleaned = [v for v in values if v not in fake_set]

    # check if anything was removed
    if len(cleaned) != len(values):
        print("⚠️ Found fake value(s)!")
        print("Original list:", values)
        print("Original doc text:", doc_text)
        print("Original title text:", title)

        print("-" * 60)

    return cleaned if cleaned else np.nan

# apply cleaning with both columns
output_data["Acid_antibiotic_normalized_cleaned"] = output_data.apply(
    lambda row: clean_list_with_logging(row["Acid_antibiotic_normalized_cleaned"], row["Doc_text"], row["Title"]),
    axis=1
)

In [38]:
output_data.columns


Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text', 'Separated_products',
       'Acid_antibiotic_normalized_cleaned'],
      dtype='object')

# Result 

In [39]:
from display_texts import compute_cleanup_metrics, product_cleanup_summary

In [40]:
metrics = compute_cleanup_metrics(output_data)
output_data = output_data.loc[:,~output_data.columns.isin(['Separated_products'])]
output_data.head()


,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Acid_antibiotic_normalized_cleaned
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,title,production of zeaxanthin and,[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,abstract,heme proteins production,[heme proteins]
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,not_found,NaN,NaN
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,abstract,ethanol production rates,[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,abstract,sterol biosynthesis,[sterol]


In [41]:
import numpy as np
import pandas as pd

def normalize_lists(x, col_name=""):
    """
    Convert lists that are empty or contain only NaN to np.nan,
    and print the original value for logging.
    """
    if isinstance(x, list):
        # Empty list
        if len(x) == 0:
            print(f"⚠️ Empty list in column '{col_name}': {x}")
            return np.nan
        # Single-element list with NaN
        if len(x) == 1 and pd.isna(x[0]):
            print(f"⚠️ [np.nan] list in column '{col_name}': {x}")
            return np.nan
    return x

# Apply to your column
output_data['Acid_antibiotic_normalized_cleaned'] = output_data['Acid_antibiotic_normalized_cleaned'].apply(
    lambda x: normalize_lists(x, col_name="Acid_antibiotic_normalized_cleaned")
)


In [42]:
check_save_file(output_data, output_file, 'Articles')

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Output/Articles/filtered_metabolic_eng_articles_with_products_cleaned_V_2025_09_30.json


In [43]:
check_again = output_data.Acid_antibiotic_normalized_cleaned.explode()

In [44]:
check_again = check_again.dropna()
check_again = check_again.loc[check_again.str.split(' ').str.len()>1]

In [45]:
check_again.loc[check_again.str.contains('titre')]

21543257    ethanol titres
Name: Acid_antibiotic_normalized_cleaned, dtype: object

In [46]:
check_again.loc[check_again.str.contains('group')]

34370382    B‐group vitamins
Name: Acid_antibiotic_normalized_cleaned, dtype: object

In [47]:
check_again.loc[check_again.str.contains('cose')]

15239699                                      glucose oxidase
20384302    uridine 5-diphospho-2-acetonyl-2-deoxy-alpha-D...
20571795                      glucose dehydrogenase gluconate
24041310                                         xsp8 glucose
25435503                                      glucose oxidase
26832825                                α-glucose 1-phosphate
27324299                            glucose-moiety precursors
29480380                    glucose 6-phosphate dehydrogenase
29864584                                       glucose growth
30083963                                      glucose oxidase
30576911                                    d-psicose ethanol
30894609                               β-glucosidases glucose
31294913                             xylose/glucose isomerase
32336091                                     guanosine fucose
32716052                          ga-3-O-glucose (2631-units)
33051008                                      glucose oxidase
38037762

In [48]:
check_again.str.split(' ').str[0].value_counts().head(60)

Acid_antibiotic_normalized_cleaned
fatty                    294
amino                     90
aromatic                  33
vitamin                   25
methyl                    21
biomass                   20
nicotinamide              15
ethanol                   14
coenzyme                  14
ethyl                     14
odd                       13
protein                   13
carbon                    13
gene                      13
polyhydroxyalkanoate      12
cyclic                    11
fuel                      10
alkaline                  10
plasmid                   10
sesquiterpene              9
benzylisoquinoline         9
lipid                      9
nylon                      9
butyl                      9
ethylene                   9
isoamyl                    8
lignocellulolytic          8
glucose                    8
cannabinoid                8
tropane                    8
cellulolytic               8
phenolic                   8
antifungal                 7
omega-3 

In [49]:
check_again_single = output_data.Acid_antibiotic_normalized_cleaned.explode()
check_again_single = check_again_single.dropna()
check_again_single = check_again_single.loc[check_again_single.str.split(' ').str.len()==1]

In [67]:
output_data.columns

Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text',
       'Acid_antibiotic_normalized_cleaned'],
      dtype='object')

In [68]:
output_data.dropna(subset=['Acid_antibiotic_normalized_cleaned']).Product_Source.value_counts()

Product_Source
title        8278
abstract     4836
full_text    1341
not_found       0
Name: count, dtype: int64

# Summary

In [64]:
from IPython.display import display, HTML
from display_texts import compute_cleanup_metrics, product_cleanup_summary

In [65]:
result_text = product_cleanup_summary(metrics, output_file)
display(HTML(result_text))